# LangGraph and LangSmith - Agentic RAG Powered by LangChain

In the following notebook we'll complete the following tasks:

- 🤝 Breakout Room #1:
  1. Install required libraries
  2. Set Environment Variables
  3. Creating our Tool Belt
  4. Creating Our State
  5. Creating and Compiling A Graph!

- 🤝 Breakout Room #2:
  1. Evaluating the LangGraph Application with LangSmith
  2. Adding Helpfulness Check and "Loop" Limits
  3. LangGraph for the "Patterns" of GenAI

# 🤝 Breakout Room #1

## Part 1: LangGraph - Building Cyclic Applications with LangChain

LangGraph is a tool that leverages LangChain Expression Language to build coordinated multi-actor and stateful applications that includes cyclic behaviour.

### Why Cycles?

In essence, we can think of a cycle in our graph as a more robust and customizable loop. It allows us to keep our application agent-forward while still giving the powerful functionality of traditional loops.

Due to the inclusion of cycles over loops, we can also compose rather complex flows through our graph in a much more readable and natural fashion. Effectively allowing us to recreate application flowcharts in code in an almost 1-to-1 fashion.

### Why LangGraph?

Beyond the agent-forward approach - we can easily compose and combine traditional "DAG" (directed acyclic graph) chains with powerful cyclic behaviour due to the tight integration with LCEL. This means it's a natural extension to LangChain's core offerings!

## Task 1:  Dependencies

We'll first install all our required libraries.

> NOTE: If you're running this locally - please skip this step.

In [1]:
#!pip install -qU langchain langchain_openai langchain-community langgraph arxiv

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.8/145.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.4/412.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 3.1 MB/s eta 0:00:00


## Task 2: Environment Variables

We'll want to set both our OpenAI API key and our LangSmith environment variables.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [2]:
os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY")

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"AIE6 - LangGraph - {uuid4().hex[0:8]}"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")

## Task 3: Creating our Tool Belt

As is usually the case, we'll want to equip our agent with a toolbelt to help answer questions and add external knowledge.

There's a tonne of tools in the [LangChain Community Repo](https://github.com/langchain-ai/langchain/tree/master/libs/community/langchain_community/tools) but we'll stick to a couple just so we can observe the cyclic nature of LangGraph in action!

We'll leverage:

- [Tavily Search Results](https://github.com/langchain-ai/langchain/blob/master/libs/community/langchain_community/tools/tavily_search/tool.py)
- [Arxiv](https://github.com/langchain-ai/langchain/tree/master/libs/community/langchain_community/tools/arxiv)

#### 🏗️ Activity #1:

Please add the tools to use into our toolbelt.

> NOTE: Each tool in our toolbelt should be a method.

In [4]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools.arxiv.tool import ArxivQueryRun

tavily_tool = TavilySearchResults(max_results=5)

tool_belt = [
    tavily_tool,
    ArxivQueryRun(),
]

### Model

Now we can set-up our model! We'll leverage the familiar OpenAI model suite for this example - but it's not *necessary* to use with LangGraph. LangGraph supports all models - though you might not find success with smaller models - as such, they recommend you stick with:

- OpenAI's GPT-3.5 and GPT-4
- Anthropic's Claude
- Google's Gemini

> NOTE: Because we're leveraging the OpenAI function calling API - we'll need to use OpenAI *for this specific example* (or any other service that exposes an OpenAI-style function calling API.

In [5]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o", temperature=0)

Now that we have our model set-up, let's "put on the tool belt", which is to say: We'll bind our LangChain formatted tools to the model in an OpenAI function calling format.

In [6]:
model = model.bind_tools(tool_belt)

#### ❓ Question #1:

How does the model determine which tool to use?

##### ✅ Answer: 
The model determines which tool to use through OpenAI's function calling API, which allows the model to programmatically choose and execute functions. When we bind tools using model.bind_tools(), each tool gets converted into a JSON schema that defines:

1. The function name
2. A description of what the function does
3. Required and optional parameters with their types
4. Expected output format

For example, TavilySearchResults would be represented as:
```
{
  "name": "tavily_search",
  "description": "Search the web for current information",
  "parameters": {
    "query": {"type": "string", "description": "The search query"},
    "max_results": {"type": "integer", "description": "Max number of results"}
  }
}
```

When the model receives a query, it:
1. Analyzes the user's request
2. Reviews all available function schemas
3. Outputs a "function_call" object specifying which function to use and with what parameters
4. The selected function is then executed with those parameters

This structured approach ensures the model can make informed decisions about tool selection based on the query's requirements.


## Task 4: Putting the State in Stateful

Earlier we used this phrasing:

`coordinated multi-actor and stateful applications`

So what does that "stateful" mean?

To put it simply - we want to have some kind of object which we can pass around our application that holds information about what the current situation (state) is. Since our system will be constructed of many parts moving in a coordinated fashion - we want to be able to ensure we have some commonly understood idea of that state.

LangGraph leverages a `StatefulGraph` which uses an `AgentState` object to pass information between the various nodes of the graph.

There are more options than what we'll see below - but this `AgentState` object is one that is stored in a `TypedDict` with the key `messages` and the value is a `Sequence` of `BaseMessages` that will be appended to whenever the state changes.

Let's think about a simple example to help understand exactly what this means (we'll simplify a great deal to try and clearly communicate what state is doing):

1. We initialize our state object:
  - `{"messages" : []}`
2. Our user submits a query to our application.
  - New State: `HumanMessage(#1)`
  - `{"messages" : [HumanMessage(#1)}`
3. We pass our state object to an Agent node which is able to read the current state. It will use the last `HumanMessage` as input. It gets some kind of output which it will add to the state.
  - New State: `AgentMessage(#1, additional_kwargs {"function_call" : "WebSearchTool"})`
  - `{"messages" : [HumanMessage(#1), AgentMessage(#1, ...)]}`
4. We pass our state object to a "conditional node" (more on this later) which reads the last state to determine if we need to use a tool - which it can determine properly because of our provided object!

In [7]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

## Task 5: It's Graphing Time!

Now that we have state, and we have tools, and we have an LLM - we can finally start making our graph!

Let's take a second to refresh ourselves about what a graph is in this context.

Graphs, also called networks in some circles, are a collection of connected objects.

The objects in question are typically called nodes, or vertices, and the connections are called edges.

Let's look at a simple graph.

![image](https://i.imgur.com/2NFLnIc.png)

Here, we're using the coloured circles to represent the nodes and the yellow lines to represent the edges. In this case, we're looking at a fully connected graph - where each node is connected by an edge to each other node.

If we were to think about nodes in the context of LangGraph - we would think of a function, or an LCEL runnable.

If we were to think about edges in the context of LangGraph - we might think of them as "paths to take" or "where to pass our state object next".

Let's create some nodes and expand on our diagram.

> NOTE: Due to the tight integration with LCEL - we can comfortably create our nodes in an async fashion!

In [8]:
from langgraph.prebuilt import ToolNode

def call_model(state):
  messages = state["messages"]
  response = model.invoke(messages)
  return {"messages" : [response]}

tool_node = ToolNode(tool_belt)

Now we have two total nodes. We have:

- `call_model` is a node that will...well...call the model
- `tool_node` is a node which can call a tool

Let's start adding nodes! We'll update our diagram along the way to keep track of what this looks like!


In [9]:
from langgraph.graph import StateGraph, END

uncompiled_graph = StateGraph(AgentState)

uncompiled_graph.add_node("agent", call_model)
uncompiled_graph.add_node("action", tool_node)

Let's look at what we have so far:

![image](https://i.imgur.com/md7inqG.png)

Next, we'll add our entrypoint. All our entrypoint does is indicate which node is called first.

In [10]:
uncompiled_graph.set_entry_point("agent")

![image](https://i.imgur.com/wNixpJe.png)

Now we want to build a "conditional edge" which will use the output state of a node to determine which path to follow.

We can help conceptualize this by thinking of our conditional edge as a conditional in a flowchart!

Notice how our function simply checks if there is a "function_call" kwarg present.

Then we create an edge where the origin node is our agent node and our destination node is *either* the action node or the END (finish the graph).

It's important to highlight that the dictionary passed in as the third parameter (the mapping) should be created with the possible outputs of our conditional function in mind. In this case `should_continue` outputs either `"end"` or `"continue"` which are subsequently mapped to the action node or the END node.

In [11]:
def should_continue(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  return END

uncompiled_graph.add_conditional_edges(
    "agent",
    should_continue
)

Let's visualize what this looks like.

![image](https://i.imgur.com/8ZNwKI5.png)

Finally, we can add our last edge which will connect our action node to our agent node. This is because we *always* want our action node (which is used to call our tools) to return its output to our agent!

In [12]:
uncompiled_graph.add_edge("action", "agent")

Let's look at the final visualization.

![image](https://i.imgur.com/NWO7usO.png)

All that's left to do now is to compile our workflow - and we're off!

In [13]:
compiled_graph = uncompiled_graph.compile()


#### ❓ Question #2:

Is there any specific limit to how many times we can cycle? 
If not, how could we impose a limit to the number of cycles?

#### ✅ *Answer:*

Currently, our graph implementation *does not have an explicit limit* on the number of cycles (agent -> action -> agent). The `should_continue` function only checks if the last message contains tool calls. If the agent continuously decides to call tools without providing a final answer, it could potentially loop indefinitely.

To impose a limit, we can:

1.  *Modify the `AgentState`:* Add a field to keep track of the number of iterations:
    ```python
    class AgentState(TypedDict):
        messages: Annotated[Sequence[BaseMessage], operator.add]
        iterations: int # Add this line
    ```
2.  *Update the `should_continue` function:* Modify the conditional logic to check the iterations counter. If the number of iterations exceeds a predefined max, it should return `END`, regardless of whether the agent wants to call another tool.
    ```python
    MAX_ITERATIONS = 5

    def should_continue(state: AgentState):
        messages = state['messages']
        last_message = messages[-1]
        # We retrieve the current iteration count, defaulting to 0 if not present
        iterations = state.get('iterations', 0)

        # We increment the iteration count before the check
        iterations += 1

        # Store the updated count back in the state.
        # Note: Modifying state directly in conditional edges is possible but a cleaner approach is to have a dedicated node update the count. For simplicity here, let us do it inline.
        state['iterations'] = iterations

        # If we have tool calls and haven't exceeded the limit, continue to action
        if last_message.tool_calls and iterations < MAX_ITERATIONS:
            return "action"
        # In all other cases, finish
        return END
    ```
3.  *Initialize the counter:* Ensure the `iterations` field is initialized (to 0) when the graph starts. This will involve adjusting how the initial state is handled within the nodes/edges. When invoking the graph, we will need to ensure the initial input dictionary includes `"iterations": 0`.

These 3 changes ensure the agent cannot get stuck in an infinite loop.

## Using Our Graph

Now that we've created and compiled our graph - we can call it *just as we'd call any other* `Runnable`!

Let's try out a few examples to see how it fairs:

In [14]:
from langchain_core.messages import HumanMessage

inputs = {"messages" : [HumanMessage(content="Who is the current captain of the Winnipeg Jets?")]}

async for chunk in compiled_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Hm8Lxyk5oxiBnx7J3HPLxg6b', 'function': {'arguments': '{"query":"current captain of the Winnipeg Jets 2023"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 162, 'total_tokens': 189, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f5bdcc3276', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-071d6046-d6f1-401c-bb65-af0f05f9124b-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current captain of the Winnipeg Jets 2023'}, 'id': 'call_Hm8Lxyk5oxiBnx7J3HPLxg6b', 'type': 'tool_call'}], usage_metadata={'input_tokens': 162, 'output_t

Let's look at what happened:

1. Our state object was populated with our request
2. The state object was passed into our entry point (agent node) and the agent node added an `AIMessage` to the state object and passed it along the conditional edge
3. The conditional edge received the state object, found the "tool_calls" `additional_kwarg`, and sent the state object to the action node
4. The action node added the response from the OpenAI function calling endpoint to the state object and passed it along the edge to the agent node
5. The agent node added a response to the state object and passed it along the conditional edge
6. The conditional edge received the state object, could not find the "tool_calls" `additional_kwarg` and passed the state object to END where we see it output in the cell above!

Now let's look at an example that shows a multiple tool usage - all with the same flow!

In [15]:
inputs = {"messages" : [HumanMessage(content="Search Arxiv for the QLoRA paper, then search each of the authors to find out their latest Tweet using Tavily!")]}

async for chunk in compiled_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        if node == "action":
          print(f"Tool Used: {values['messages'][0].name}")
        print(values["messages"])

        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_QfKpAPIhA1I9ywy2B1YryQeU', 'function': {'arguments': '{"query":"QLoRA"}', 'name': 'arxiv'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 178, 'total_tokens': 195, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f5bdcc3276', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-6650e6de-585b-4df1-8f65-ca42ae582084-0', tool_calls=[{'name': 'arxiv', 'args': {'query': 'QLoRA'}, 'id': 'call_QfKpAPIhA1I9ywy2B1YryQeU', 'type': 'tool_call'}], usage_metadata={'input_tokens': 178, 'output_tokens': 17, 'total_tokens': 195, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'a

#### 🏗️ Activity #2:

Please write out the steps the agent took to arrive at the correct 
answer.

✅ *Answer:*

###### 1. *Input:* The initial `HumanMessage` ("Search Arxiv for the QLoRA paper, then search each of the authors to find out their latest Tweet using Tavily!") is added to the state.

###### 2. *Agent Node:* The agent node receives the state. The LLM determines that it needs to use the `arxiv` tool to find the QLoRA paper and its authors. It adds an `AIMessage` with a `tool_calls` attribute requesting the use of the `arxiv` tool to the state.

###### 3. *Conditional Edge:* The edge checks the latest message in the state. Since it contains `tool_calls`, it routes the state to the `action` node.

###### 4. *Action Node:* This node executes the requested `arxiv` tool call. It searches Arxiv for the QLoRA paper, finds it, and extracts the authors. It adds the result as a `ToolMessage` which is added to the state.

###### 5. *Agent Node:* The state (now containing the Arxiv search results, including authors) is passed back to the agent node. The agent processes this information and determines the next step: searching for the latest tweet for each author needs use of the `tavily_search_results_json` tool. It adds another `AIMessage` to the state, this time with multiple `tool_calls` (one for each author's tweet search).

###### 6. *Conditional Edge:* Again, the edge detects `tool_calls` in the latest message and routes the state to the `action` node.

###### 7. *Action Node:* The action node executes each of the requested `tavily_search_results_json` tool calls sequentially. It adds the results (latest tweets or relevant info found by Tavily) as `ToolMessage`s to the state.

###### 8. *Agent Node:* The state (now containing Arxiv results and Tavily tweet search results) is passed back to the agent node. The agent synthesizes all the gathered information into a final answer. It adds a final `AIMessage` containing this answer to the state. This message does *not* contain `tool_calls`.

###### 9. *Conditional Edge:* The edge checks the latest message. Since it does *not* contain `tool_calls`, it routes the state to the `END` node.

###### 10. *Output:* The graph execution finishes, and the final state, including the agent's synthesized answer, is returned.

# 🤝 Breakout Room #2

## Part 1: LangSmith Evaluator

### Pre-processing for LangSmith

To do a little bit more preprocessing, let's wrap our LangGraph agent in a simple chain.

In [16]:
def convert_inputs(input_object):
  return {"messages" : [HumanMessage(content=input_object["question"])]}

def parse_output(input_state):
  return input_state["messages"][-1].content

agent_chain = convert_inputs | compiled_graph | parse_output

In [17]:
agent_chain.invoke({"question" : "What is RAG?"})

"RAG stands for Retrieval-Augmented Generation. It is a technique used in natural language processing (NLP) that combines retrieval-based methods with generative models to improve the quality and accuracy of generated text. Here's how it generally works:\n\n1. **Retrieval**: The system first retrieves relevant information from a large corpus or database. This step involves searching for documents, passages, or data that are pertinent to the input query or context.\n\n2. **Augmentation**: The retrieved information is then used to augment the input to a generative model. This means that the generative model has access to additional context or facts that can help it produce more accurate and informative responses.\n\n3. **Generation**: Finally, the generative model uses both the original input and the retrieved information to generate a response. This can be in the form of answering questions, completing sentences, or creating more complex text outputs.\n\nRAG is particularly useful in sc

### Task 1: Creating An Evaluation Dataset

Just as we saw last week, we'll want to create a dataset to test our Agent's ability to answer questions.

In order to do this - we'll want to provide some questions and some answers. Let's look at how we can create such a dataset below.

```python
questions = [
    "What optimizer is used in QLoRA?",
    "What data type was created in the QLoRA paper?",
    "What is a Retrieval Augmented Generation system?",
    "Who authored the QLoRA paper?",
    "What is the most popular deep learning framework?",
    "What significant improvements does the LoRA system make?"
]

answers = [
    {"must_mention" : ["paged", "optimizer"]},
    {"must_mention" : ["NF4", "NormalFloat"]},
    {"must_mention" : ["ground", "context"]},
    {"must_mention" : ["Tim", "Dettmers"]},
    {"must_mention" : ["PyTorch", "TensorFlow"]},
    {"must_mention" : ["reduce", "parameters"]},
]
```

#### 🏗️ Activity #3:

Please create a dataset in the above format with at least 5 questions.

In [18]:
questions = [
    "What optimizer is used in QLoRA?",
    "What data type was created in the QLoRA paper?",
    "What is a Retrieval Augmented Generation system?",
    "Who authored the QLoRA paper?",
    "What is the most popular deep learning framework?",
    "What significant improvements does the LoRA system make?"
]

answers = [
    {"must_mention" : ["paged", "optimizer"]},
    {"must_mention" : ["NF4", "NormalFloat"]},
    {"must_mention" : ["ground", "context"]},
    {"must_mention" : ["Tim", "Dettmers"]},
    {"must_mention" : ["PyTorch", "TensorFlow"]},
    {"must_mention" : ["reduce", "parameters"]},
]

Now we can add our dataset to our LangSmith project using the following code which we saw last Thursday!

In [19]:
from langsmith import Client

client = Client()

dataset_name = f"Retrieval Augmented Generation - Evaluation Dataset - {uuid4().hex[0:8]}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Questions about the QLoRA Paper to Evaluate RAG over the same paper."
)

client.create_examples(
    inputs=[{"question" : q} for q in questions],
    outputs=answers,
    dataset_id=dataset.id,
)

{'example_ids': ['061af5be-8448-4033-ab23-be88ec7783f8',
  'e09768bf-c0f8-4c75-ba1f-b3415927737f',
  '16e5cc24-5eca-4c8c-8468-c9a6df4991f3',
  'daf190cc-a225-4005-8737-ca301f8e9c06',
  '1b40fabe-36c7-475c-91a1-0b0916576383',
  '476f806d-1f66-45d5-951a-6c5d4f21e44d'],
 'count': 6}

#### ❓ Question #3:

How are the correct answers associated with the questions?

> NOTE: Feel free to indicate if this is problematic or not

✅ *Answer:*

The correct answers (`answers` list) are associated with the questions (`questions` list) based on their *order* or *index*. 

The first question in the `questions` list corresponds to the first dictionary in the `answers` list, the second question corresponds to the second answer dictionary, and so on.

> This positional association is simple but can be problematic. If the order of elements in either list is changed independently, or if elements are added/removed without ensuring the corresponding element in the other list is also adjusted correctly, the question-answer pairings will become incorrect. This makes the association fragile. A more robust method might involve using a list of dictionaries, where each dictionary contains both the question and its corresponding answer criteria, e.g., `[{"question": "...", "must_mention": [...]}, ...]`.

### Task 2: Adding Evaluators

Now we can add a custom evaluator to see if our responses contain the expected information.

We'll be using a fairly naive exact-match process to determine if our response contains specific strings.

In [20]:
from langsmith.evaluation import EvaluationResult, run_evaluator

@run_evaluator
def must_mention(run, example) -> EvaluationResult:
    prediction = run.outputs.get("output") or ""
    required = example.outputs.get("must_mention") or []
    score = all(phrase in prediction for phrase in required)
    return EvaluationResult(key="must_mention", score=score)

#### ❓ Question #4:

What are some ways you could improve this metric as-is?

> NOTE: Alternatively you can suggest where gaps exist in this method.


 ✅ *Answer:*

 The current `must_mention` metric relies on a simple, exact substring match. Here are some gaps and potential improvements:

 **Gaps:**

 1.  **Case Sensitivity:** The check `phrase in prediction` is case-sensitive. "LangGraph" would not match "langgraph".
 2.  **Exact Phrasing:** It requires the *exact* string. Synonyms, paraphrases, or slightly different but semantically equivalent phrasings won't be detected (e.g., "retrieval augmented generation" vs. "generation augmented by retrieval").
 3.  **Word Forms:** Different forms of a word (e.g., "retrieve", "retrieval", "retrieved") won't match unless all forms are explicitly listed in `must_mention`.
 4.  **Context Ignorance:** The metric only checks for presence, not for correct usage or context. A phrase could be mentioned in a way that contradicts the intended meaning or is irrelevant.
 5.  **All-or-Nothing Scoring:** The `all()` function means if even one phrase is missing, the score is 0 (False). It doesn't give partial credit if most phrases are present.
 6.  **Substring Ambiguity:** A required phrase might appear as part of a larger, unrelated word (e.g., requiring "graph" might match "graphite").

 **Potential Improvements:**

 1.  **Case Insensitivity:** Convert both `prediction` and `phrase` to lowercase before checking: `phrase.lower() in prediction.lower()`.
 2.  **Stemming/Lemmatization:** Process both the prediction and phrases to reduce words to their root form before comparison.
 3.  **Regular Expressions:** Use regex with word boundaries (`\bphrase\b`) to avoid partial word matches and potentially handle variations.
 4.  **Partial Scoring:** Calculate a score based on the *proportion* of required phrases found (e.g., `sum(phrase in prediction for phrase in required) / len(required)`).
 5.  **Semantic Similarity:** Use embedding models to compare the semantic similarity between the required concepts and the prediction, rather than relying on exact string matches.
 6.  **LLM-as-Judge:** Use another LLM to evaluate if the prediction contains the information conveyed by the required phrases, allowing for more nuanced understanding of synonyms and context.

Task 3: Evaluating

All that is left to do is evaluate our agent's response!

In [21]:
experiment_results = client.evaluate(
    agent_chain,
    data=dataset_name,
    evaluators=[must_mention],
    experiment_prefix=f"RAG Pipeline - Evaluation - {uuid4().hex[0:4]}",
    metadata={"version": "1.0.0"},
)

View the evaluation results for experiment: 'RAG Pipeline - Evaluation - 0d5d-88fb192f' at:
https://smith.langchain.com/o/39f31aed-fcde-5ca7-a4f1-a8447f10d513/datasets/1c554a47-0d4d-4c4a-a9d8-00b34572bbd0/compare?selectedSessions=1500cbfb-570e-44bc-9db8-3e5daf3c37e8




0it [00:00, ?it/s]

In [23]:
experiment_results

,inputs.question,outputs.output,error,reference.must_mention,feedback.must_mention,execution_time,example_id,id
0,What data type was created in the QLoRA paper?,The QLoRA paper introduced a novel data type c...,None,"[NF4, NormalFloat]",True,6.471813,1bd0e1ac-8002-4ae7-91a2-32471ca671cf,dfb902f1-9571-43fe-983c-8a14dc6b33cf
1,What is a Retrieval Augmented Generation system?,A Retrieval Augmented Generation (RAG) system ...,None,"[ground, context]",True,3.509352,255a1a4b-765c-40d5-9e78-4575b6683885,53a328e2-f697-42b7-823c-e5ba12ef6889
2,What optimizer is used in QLoRA?,"QLoRA uses ""paged optimizers"" for tuning large...",None,"[paged, optimizer]",True,4.620683,61922d3f-057a-48d9-b5f9-6356dcce4bc0,99019c78-69eb-4b45-8ebf-287dedbc8c42
3,What is the most popular deep learning framework?,"In 2023, the most popular deep learning framew...",None,"[PyTorch, TensorFlow]",True,4.486045,6993df54-1a1a-442a-a469-e49cbc25e940,15b8012e-746a-49ec-8269-1943a92e784c
4,What significant improvements does the LoRA sy...,"The LoRa system, specifically LoRaWAN, has see...",None,"[reduce, parameters]",False,7.493263,9097f7b1-7bff-4131-a824-aa8e6527447a,d7828f99-a429-4478-8740-17d1c9fa78aa
5,Who authored the QLoRA paper?,"The QLoRA paper was authored by Tim Dettmers, ...",None,"[Tim, Dettmers]",True,4.963734,f833ee30-a5a4-479e-847d-b9ada317c831,f8769a67-712a-43aa-9ba1-b9abdd07462b


## Part 2: LangGraph with Helpfulness:

### Task 3: Adding Helpfulness Check and "Loop" Limits

Now that we've done evaluation - let's see if we can add an extra step where we review the content we've generated to confirm if it fully answers the user's query!

We're going to make a few key adjustments to account for this:

1. We're going to add an artificial limit on how many "loops" the agent can go through - this will help us to avoid the potential situation where we never exit the loop.
2. We'll add to our existing conditional edge to obtain the behaviour we desire.

First, let's define our state again - we can check the length of the state object, so we don't need additional state for this.

In [22]:
class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

Now we can set our graph up! This process will be almost entirely the same - with the inclusion of one additional node/conditional edge!

#### 🏗️ Activity #5:

Please write markdown for the following cells to explain what each is doing.

##### Explanation of Graph Initialization and Node Addition

The cell below initializes a new LangGraph `StateGraph` instance named `graph_with_helpfulness_check`.

1.  *`StateGraph(AgentState)`*: We create the graph object, specifying `AgentState` as the structure that will hold the state information (like messages) as it flows through the graph.
2.  *`add_node("agent", call_model)`*: We define the first node in our graph, named `"agent"`. This node is associated with the `call_model` function. When the graph execution reaches this node, it will invoke `call_model` to interact with the language model.
3.  *`add_node("action", tool_node)`*: We define a second node, named `"action"`. This node is linked to the `tool_node` function. If the agent decides to use a tool, the graph will route the execution to this node, which will then handle the tool execution.

In [23]:
graph_with_helpfulness_check = StateGraph(AgentState)

graph_with_helpfulness_check.add_node("agent", call_model)
graph_with_helpfulness_check.add_node("action", tool_node)

##### Explanation of Setting the Entry Point

The cell below sets the entry point for the `graph_with_helpfulness_check`.

*   `set_entry_point("agent")`: This command tells LangGraph that whenever the graph execution starts, it should begin at the node named `"agent"`. This is the node we previously defined to call the language model.

In [24]:
graph_with_helpfulness_check.set_entry_point("agent")

##### Explanation of Conditional Edge Logic (`tool_call_or_helpful`)

The cell below defines a Python function named `tool_call_or_helpful` which acts as the logic for a *conditional edge* in our LangGraph. This function determines the next step after the main agent node (`"agent"`) has executed.

1.  **Check for Tool Calls:** It first examines the most recent message in the `state`. If this message contains `tool_calls`, it means the agent decided to use a tool. In this case, the function returns the string `"action"`, directing the graph to proceed to the `"action"` node (which runs `tool_node`).
2.  **Check for Conversation Length:** As a safety measure, if the conversation history (`state["messages"]`) exceeds 10 messages, it returns `"END"` to stop the graph execution and prevent potential infinite loops.
3.  **Helpfulness Check (if no tool call and not too long):**
    *   If no tool call was made and the conversation isn't too long, the function proceeds to evaluate if the agent's last response was helpful.
    *   It extracts the initial user query (`messages[0]`) and the agent's final response (`messages[-1]`).
    *   It defines a `PromptTemplate` specifically asking a separate LLM (in this case, `gpt-4`) to judge the helpfulness of the `final_response` given the `initial_query`, expecting a 'Y' (Yes) or 'N' (No) answer.
    *   It creates a simple LangChain Expression Language (LCEL) chain (`prompt | model | parser`) to execute this helpfulness check.
    *   It invokes this chain with the query and response.
4.  **Routing Based on Helpfulness:**
    *   If the helpfulness check returns a response containing 'Y', the function returns `"end"`, signaling that the agent's response was satisfactory and the graph execution should terminate.
    *   If the helpfulness check returns anything else (implicitly 'N'), the function returns `"continue"`. This typically directs the graph back to the `"agent"` node, prompting the agent to try generating a better response.

In essence, this function acts as a router: either go use a tool, stop because the answer is helpful (or the conversation is too long), or continue trying to get a helpful answer.

In [26]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def tool_call_or_helpful(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  initial_query = state["messages"][0]
  final_response = state["messages"][-1]

  if len(state["messages"]) > 10:
    return "END"

  prompt_template = """\
  Given an initial query and a final response, determine if the final response is extremely helpful or not. Please indicate helpfulness with a 'Y' and unhelpfulness as an 'N'.

  Initial Query:
  {initial_query}

  Final Response:
  {final_response}"""

  prompt_template = PromptTemplate.from_template(prompt_template)

  helpfulness_check_model = ChatOpenAI(model="gpt-4")

  helpfulness_chain = prompt_template | helpfulness_check_model | StrOutputParser()

  helpfulness_response = helpfulness_chain.invoke({"initial_query" : initial_query.content, "final_response" : final_response.content})

  if "Y" in helpfulness_response:
    return "end"
  else:
    return "continue"

#### 🏗️ Activity #4:

Please write what is happening in our `tool_call_or_helpful` function!

##### Explanation of `tool_call_or_helpful` Function call:

This function acts as a conditional router within our LangGraph agent. It examines the last message generated by the agent and decides the next step based on specific criteria.

In essence, this function routes the flow to either execute a tool, end the conversation due to length or perceived helpfulness, or continue the agent's generation process if the response isn't deemed helpful enough yet.

In [28]:
graph_with_helpfulness_check.add_conditional_edges(
    "agent",
    tool_call_or_helpful,
    {
        "continue" : "agent",
        "action" : "action",
        "end" : END
    }
)

##### Explanation of `add_edge("action", "agent")`:

This line explicitly connects the `action` node back to the `agent` node.
After the `action` node successfully executes a tool (like the Tavily search),
this edge ensures that the flow returns to the `agent`.
The `agent` node will then receive the tool's output as part of the message history and can decide on the next step, potentially generating a final response or calling another tool.

In [27]:
graph_with_helpfulness_check.add_edge("action", "agent")

##### Explanation of `graph_with_helpfulness_check.compile()`:

This line takes the graph structure we've defined (`graph_with_helpfulness_check`),
including all the nodes (`agent`, `action`), the entry point, the edges,
and the conditional logic (`tool_call_or_helpful`), and compiles it into an
executable `Runnable` object.
Think of it as finalizing the blueprint and building the actual machine.
The resulting `agent_with_helpfulness_check` object can now be used to process
inputs by invoking it (e.g., using `.invoke()` or `.astream()`).

In [29]:
agent_with_helpfulness_check = graph_with_helpfulness_check.compile()

##### Explanation of the Code Below:

1.  **Define Input:**
    *   `inputs = {"messages" : [HumanMessage(content="...")]}`: We create the initial input for our compiled LangGraph agent (`agent_with_helpfulness_check`). This input is a dictionary containing a list of messages, starting with a `HumanMessage` that holds the user's complex query about LoRA, Tim Dettmers, and Attention.

2.  **Stream Agent Execution:**
    *   `async for chunk in agent_with_helpfulness_check.astream(inputs, stream_mode="updates"):`: This line executes the agent asynchronously and streams the results.
        *   `astream()`: Allows us to get results incrementally as the agent works through its steps (nodes). This is useful for seeing progress in real-time, especially for longer-running tasks.
        *   `inputs`: Passes the user's query (defined above) to the agent.
        *   `stream_mode="updates"`: Specifies that we want to receive updates from *each node* in the graph as it finishes processing. This lets us observe the agent's internal state changes and message flow step-by-step.

3.  **Process Streamed Updates:**
    *   `for node, values in chunk.items():`: Each `chunk` received from the stream is a dictionary. It contains updates from one or more nodes that executed in that step. This loop iterates through the items in the chunk, unpacking the `node` name (e.g., 'agent', 'action') and the `values` (the updated state dictionary associated with that node, typically including the current list of "messages").
    *   `print(f"Receiving update from node: '{node}'")`: Prints the name of the node that generated the current update, helping us trace the execution path.
    *   `print(values["messages"])`: Prints the current state of the "messages" list *after* the update from the specific `node`. This shows how the conversation history (including agent thoughts, tool calls, tool outputs, and final answers) evolves as the agent works.
    *   `print("\n\n")`: Adds some vertical spacing in the output for better readability between updates.

In summary, this code block invokes the previously compiled agent (`agent_with_helpfulness_check`) with a specific, multi-part question. It then iterates through the asynchronous stream of updates generated during the agent's execution, printing out which node is currently active and the state of the messages at that point. This provides a detailed, real-time view of the agent's reasoning process, including any tool usage and intermediate thoughts.

In [30]:
inputs = {"messages" : [HumanMessage(content="Related to machine learning, what is LoRA? Also, who is Tim Dettmers? Also, what is Attention?")]}

async for chunk in agent_with_helpfulness_check.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_CgtXBWTTOHJvzMwDiRiPYZnC', 'function': {'arguments': '{"query": "LoRA machine learning"}', 'name': 'arxiv'}, 'type': 'function'}, {'id': 'call_Awsj1HmQxdu2Is1LWvRH0mDV', 'function': {'arguments': '{"query": "Tim Dettmers"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}, {'id': 'call_ltEKJndZZGNiUXh0mXFETv3v', 'function': {'arguments': '{"query": "Attention mechanism in machine learning"}', 'name': 'arxiv'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 177, 'total_tokens': 250, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f5bdcc3276', 'finish_reason': 'tool_calls', 'logprobs': Non

### Task 4: LangGraph for the "Patterns" of GenAI

Let's ask our system about the 4 patterns of Generative AI:

1. Prompt Engineering
2. RAG
3. Fine-tuning
4. Agents

In [31]:
patterns = ["prompt engineering", "RAG", "fine-tuning", "LLM-based agents"]

In [32]:
for pattern in patterns:
  what_is_string = f"What is {pattern} and when did it break onto the scene??"
  inputs = {"messages" : [HumanMessage(content=what_is_string)]}
  messages = agent_with_helpfulness_check.invoke(inputs)
  print(messages["messages"][-1].content)
  print("\n\n")

**Prompt Engineering: Definition**

Prompt engineering is the process of designing and refining input prompts to effectively guide the behavior of AI models. It involves structuring or crafting instructions to produce the best possible output from a generative AI model. This can include phrasing a query, specifying a style, providing relevant context, or describing a character for the AI to mimic. The goal is to improve the accuracy and effectiveness of the AI's responses, whether in generating text, images, or other digital artifacts. Prompt engineering is crucial for creating better AI-powered services, minimizing biases, and getting better results from generative AI tools.

**History of Prompt Engineering**

Prompt engineering has been around since the early days of natural language processing (NLP) and AI systems. It gained significant attention with the release of GPT-3 by OpenAI in 2020, which showcased the potential of large-scale pretrained models. This marked a watershed momen